In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("/kaggle/input/datasets/rgislikassi/aligned-data-2/aligned_data (1).parquet")
# ou le chemin exact que tu utilises

# Recalculer les 4 scores à la volée
df["tox_spread"] = df["price_deviation_bps"].abs()

df["tox_lvr"] = (
    np.sqrt(0.5)
    * df["vol_24h"]
    * (df["binance_price"].pct_change() * 10_000).abs()
)

df["tox_realized"] = (
    (df["uniswap_volume_usd"] / df["uniswap_volume_usd"].median())
    * df["tox_spread"]
)

# Pour tox_volsize — adapter selon la colonne disponible
if "uniswap_p95_swap_size_usd" in df.columns:
    swap_col = "uniswap_p95_swap_size_usd"
elif "uniswap_avg_swap_size_usd" in df.columns:
    swap_col = "uniswap_avg_swap_size_usd"
else:
    df["_swap"] = df["uniswap_volume_usd"] / df["uniswap_n_swaps"].replace(0, np.nan)
    swap_col = "_swap"

df["tox_volsize"] = np.log10(1 + df[swap_col].fillna(0)) * df["tox_spread"]

# Matrice de corrélation
tox_cols = ["tox_spread", "tox_lvr", "tox_realized", "tox_volsize"]
corr = df[tox_cols].corr(method="pearson").round(3)
print(corr.to_string())

              tox_spread  tox_lvr  tox_realized  tox_volsize
tox_spread         1.000    0.612         0.638        0.985
tox_lvr            0.612    1.000         0.431        0.555
tox_realized       0.638    0.431         1.000        0.700
tox_volsize        0.985    0.555         0.700        1.000


In [2]:
"""
Génération des 4 figures pour le papier NeurIPS 2026
Output : /kaggle/working/figures/fig_*.pdf + .png (300 DPI)
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from scipy.stats import beta as beta_dist

# ── Setup ────────────────────────────────────────────────────
FIGS = Path("/kaggle/working/figures")
FIGS.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linewidth":    0.5,
    "figure.dpi":        300,
})

# ── Données ──────────────────────────────────────────────────
CONFIGS = ["R1\nbaseline", "R2\nlvr", "R3\nspread", "R4\nrealized",
           "R5\nvolsize",  "R6\nlvr+spread", "R7\nall"]
LABELS  = ["R1 baseline", "R2 lvr", "R3 spread", "R4 realized",
           "R5 volsize",  "R6 lvr+spread", "R7 all"]
N       = 15
R5_IDX  = 4

cvar10  = [-11.84, -14.12, -26.33, -10.61,   4.00, -27.31, -63.09]
cvar25  = [ -5.46,  -4.00, -14.23,  -4.81,   5.87, -20.30, -42.59]
wins    = [13, 14, 11, 12, 15, 8, 10]
win_pct = [w / N * 100 for w in wins]

regime_data = {
    "Calm":     [20.88, 36.78, 12.54, 30.15, 20.25, 15.19,  -0.66],
    "Normal":   [32.82, 38.82, 13.34, 34.93, 31.19, 12.73,  29.53],
    "Stressed": [11.87, 45.16, 10.53, 62.48, 48.52, 20.39,  62.33],
}

corr_matrix = np.array([
    [1.000, 0.612, 0.638, 0.985],
    [0.612, 1.000, 0.431, 0.555],
    [0.638, 0.431, 1.000, 0.700],
    [0.985, 0.555, 0.700, 1.000],
])
TOX_LABELS = ["spread", "lvr", "realized", "volsize"]

GREEN = "#27ae60"
RED   = "#e74c3c"
BLUE  = "#2980b9"
GRAY  = "#95a5a6"


# ── Helpers ──────────────────────────────────────────────────
def cp_ci(k, n, alpha=0.05):
    lo = float(beta_dist.ppf(alpha / 2,     max(k, 1e-9), n - k + 1)) if k > 0 else 0.0
    hi = float(beta_dist.ppf(1 - alpha / 2, k + 1, max(n - k, 1e-9))) if k < n else 1.0
    return lo, hi


def save(fig, name):
    fig.savefig(FIGS / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(FIGS / f"{name}.png", bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"   ✅ {name}.pdf / .png")


# ============================================================
# FIG 1 — CVaR10% et CVaR25%
# ============================================================
fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(CONFIGS))
w = 0.35

cols10 = [GREEN if v >= 0 else RED       for v in cvar10]
cols25 = [GREEN if v >= 0 else "#f1948a" for v in cvar25]

ax.bar(x - w / 2, cvar10, w, color=cols10,
       label="CVaR$_{10\\%}$", zorder=3,
       edgecolor="white", linewidth=0.4)
ax.bar(x + w / 2, cvar25, w, color=cols25,
       label="CVaR$_{25\\%}$", zorder=3,
       edgecolor="white", linewidth=0.4, alpha=0.8)

ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.6)

ax.set_xticks(x)
ax.set_xticklabels(CONFIGS, fontsize=8.5)
ax.set_ylabel("Excess capital vs passive (\\$)")
ax.set_xlabel("Configuration")
ax.set_title("CVaR of excess capital over passive baseline")
ax.legend(framealpha=0.9, fontsize=8.5, loc="lower left")
fig.tight_layout()
save(fig, "fig1_cvar")


# ============================================================
# FIG 2 — Win rate + Clopper-Pearson CI
# ============================================================
fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(CONFIGS))

cis    = [cp_ci(k, N) for k in wins]
lo_err = [wp / 100 - ci[0] for wp, ci in zip(win_pct, cis)]
hi_err = [ci[1] - wp / 100 for wp, ci in zip(win_pct, cis)]

cols_wr = []
for wp in win_pct:
    if wp == 100:
        cols_wr.append(GREEN)
    elif wp >= 80:
        cols_wr.append(BLUE)
    elif wp >= 67:
        cols_wr.append(GRAY)
    else:
        cols_wr.append(RED)

ax.bar(x, win_pct, color=cols_wr,
       zorder=3, edgecolor="white", linewidth=0.4, width=0.6)

ax.errorbar(
    x, win_pct,
    yerr=[[e * 100 for e in lo_err], [e * 100 for e in hi_err]],
    fmt="none", color="black",
    capsize=4, linewidth=1.2, capthick=1.2, zorder=4,
)

ax.axhline(100, color=GREEN, linewidth=0.8, linestyle=":",
           alpha=0.7, label="100\\% (perfect)")
ax.axhline(win_pct[0], color=GRAY, linewidth=0.8, linestyle="--",
           alpha=0.7, label=f"R1 baseline ({win_pct[0]:.0f}\\%)")

# Annoter R5 avec texte simple (pas de flèche)
ax.text(x[R5_IDX], win_pct[R5_IDX] + 2, "100\\%",
        ha="center", va="bottom",
        fontsize=8.5, color=GREEN, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(CONFIGS, fontsize=8.5)
ax.set_ylabel("Win rate vs passive (\\%)")
ax.set_xlabel("Configuration")
ax.set_ylim(0, 115)
ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda v, _: f"{v:.0f}%")
)
ax.set_title("Episode-level win rate with 95\\% Clopper-Pearson confidence intervals")
ax.legend(framealpha=0.9, fontsize=8.5, loc="lower right")
fig.tight_layout()
save(fig, "fig2_winrate")


# ============================================================
# FIG 3 — Heatmap performance par régime
# ============================================================
fig, ax = plt.subplots(figsize=(6.5, 4.8))

regimes  = list(regime_data.keys())
data_mat = np.array([regime_data[r] for r in regimes]).T  # shape (7, 3)

cmap = LinearSegmentedColormap.from_list(
    "rg", ["#e74c3c", "#f9e79f", "#27ae60"], N=256
)
vmin, vmax = data_mat.min(), data_mat.max()

im = ax.imshow(data_mat, cmap=cmap, aspect="auto", vmin=vmin, vmax=vmax)
cb = plt.colorbar(im, ax=ax, label="Mean excess capital (\\$)", shrink=0.85)
cb.ax.tick_params(labelsize=8)

ax.set_xticks(range(3))
ax.set_xticklabels(regimes, fontsize=9)
ax.set_yticks(range(7))
ax.set_yticklabels(LABELS, fontsize=8.5)

for i in range(7):
    for j in range(3):
        v      = data_mat[i, j]
        norm_v = (v - vmin) / (vmax - vmin)
        tc     = "white" if (norm_v > 0.65 or norm_v < 0.30) else "black"
        weight = "bold" if i == R5_IDX else "normal"
        ax.text(j, i, f"\\${v:.1f}",
                ha="center", va="center",
                fontsize=8, color=tc, fontweight=weight)

# Encadrer R5 (sobre, sans flèche)
rect = plt.Rectangle((-0.5, R5_IDX - 0.5), 3, 1,
                      fill=False, edgecolor="black", linewidth=2)
ax.add_patch(rect)

ax.set_title("Mean excess capital (\\$) by dominant market regime")
fig.tight_layout()
save(fig, "fig3_regime")


# ============================================================
# FIG 4 — Matrice de corrélation
# ============================================================
fig, ax = plt.subplots(figsize=(5.5, 4.8))

cmap2 = LinearSegmentedColormap.from_list(
    "corr", ["#ffffff", "#2980b9"], N=256
)
im2 = ax.imshow(corr_matrix, cmap=cmap2, vmin=0.4, vmax=1.0, aspect="auto")
cb2 = plt.colorbar(im2, ax=ax, label="Pearson r", shrink=0.85)
cb2.ax.tick_params(labelsize=8)

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(TOX_LABELS, fontsize=9, rotation=20, ha="right")
ax.set_yticklabels(TOX_LABELS, fontsize=9)

for i in range(4):
    for j in range(4):
        v    = corr_matrix[i, j]
        diag = i == j
        lbl  = "—" if diag else f"{v:.3f}"
        tc   = "white" if (not diag and v > 0.80) else ("gray" if diag else "black")
        weight = "bold" if (not diag and v > 0.90) else "normal"
        ax.text(j, i, lbl,
                ha="center", va="center",
                fontsize=9, color=tc, fontweight=weight)

ax.set_title("Pairwise Pearson correlations between toxicity scores")
fig.tight_layout()
save(fig, "fig4_corr")


# ── Résumé ───────────────────────────────────────────────────
print(f"\n✅ 4 figures dans {FIGS}/")
print("   fig1_cvar.pdf / .png")
print("   fig2_winrate.pdf / .png")
print("   fig3_regime.pdf / .png")
print("   fig4_corr.pdf / .png")

   ✅ fig1_cvar.pdf / .png
   ✅ fig2_winrate.pdf / .png
   ✅ fig3_regime.pdf / .png
   ✅ fig4_corr.pdf / .png

✅ 4 figures dans /kaggle/working/figures/
   fig1_cvar.pdf / .png
   fig2_winrate.pdf / .png
   fig3_regime.pdf / .png
   fig4_corr.pdf / .png
